# Excutive Summary

**Enhancement of Report Automation**

> Background

Regular performance reporting (such as quarterly KPI metrics, server uptime, bug tracking, and CSAT scores) is crucial for executive decision-making. However, manually collecting data, formatting slides, generating PDFs, and distributing reports via email is repetitive, time-consuming, and prone to human error.

While the current Python prototype successfully establishes an end-to-end basic reporting workflow (Data Setup $\rightarrow$ Presentation Design $\rightarrow$ PDF Export $\rightarrow$ Email Distribution), scaling this into an enterprise-ready system requires eliminating manual data entry, providing intelligent insights, and making the tool easily accessible to non-technical stakeholders.

> Objective

The primary goal of this project expansion is to transform the foundational script into a fully automated, scalable, and intelligent Reporting-as-a-Service (RaaS) pipeline.

Key objectives include: Automation, Intelligence, Accessibility, Reliability & Scalability

> Core Features

1. Dynamic Data Sourcing (example Direct Database Queries, API Integration, Cloud Spreadsheet Support)

2. Native Data Visualization (Generate editable PPTX bar charts, line graphs, and pie charts programmatically via python-pptx. - Create custom trend charts using matplotlib or seaborn and inject them as high-resolution visual components into the presentation)

3. AI-Generated Executive Summaries (Utilize Gemini API or others LLM to analyze raw performance tables and generate concise 3-bullet-point executive narratives explaining metric trends and variances - Automatically format and insert these AI insights into dedicated text containers within the presentation slides.

4. Automated Pipeline & Scheduling (Schedule regular automated report runs (e.g., every Monday at 08:00 AM) using GitHub Actions, etc)

5. Multi-Channel Distribution & Cloud Archiving (Send instant summary cards with direct PDF download links to Email, Slack, or Microsoft Teams channels via Webhooks)

# Execution

## Set-up Environment

In [1]:
!apt-get update && apt-get install -y libreoffice
!pip install python-pptx
!pip install google-genai

Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,013 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.2 MB]
Get:11 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,544 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,590 kB]
Get:14 htt

In [12]:
import os
import sys
import platform
import subprocess
import smtplib
import json
import re
import time
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from getpass import getpass
import io

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.enum.shapes import MSO_SHAPE
from pptx.chart.data import CategoryChartData
from pptx.enum.chart import XL_CHART_TYPE, XL_LEGEND_POSITION, XL_LABEL_POSITION
from google import genai
from google.genai import types

# Dark Theme (Untuk Tabel Performa)
COLOR_BG_DARK = RGBColor(30, 41, 59)          # Dark Slate
COLOR_CARD_DARK = RGBColor(15, 23, 42)        # Very Dark Slate
COLOR_TEXT_LIGHT = RGBColor(241, 245, 249)   # Off-white
COLOR_TEXT_MUTED = RGBColor(148, 163, 184)   # Gray
COLOR_ACCENT_BLUE = RGBColor(59, 130, 246)   # Blue Accent

# Light Theme (Untuk Visual Dashboard ala Power BI)
COLOR_BG_LIGHT = RGBColor(248, 250, 252)     # Off-White / Soft Gray
COLOR_CARD_LIGHT = RGBColor(255, 255, 255)   # White
COLOR_TEXT_DARK = RGBColor(15, 23, 42)       # Dark Slate Text
COLOR_ACCENT_POWERBI = RGBColor(14, 165, 233) # Power BI Bright Blue

# Status Colors
COLOR_GREEN = RGBColor(34, 197, 94)       # Sukses (Hijau)
COLOR_RED = RGBColor(239, 68, 68)         # Merah

## Define Function

### Load Data

In [3]:
def load_and_process_car_sales_data(url_or_path):
    """
    Mengunduh dataset dari GitHub/Lokal dan memprosesnya menggunakan Pandas.
    Mengembalikan dataframe mentah serta data teragregasi.
    """
    raw_url = url_or_path.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")

    print(f"Membaca data dari: {raw_url}")
    try:
        df = pd.read_csv(raw_url)
        print(f"✅ Data berhasil dimuat! Total baris: {len(df):,}")
    except Exception as e:
        print(f"❌ Gagal membaca data dari URL/File: {e}")
        raise e

    # Normalisasi nama kolom
    df.columns = df.columns.str.strip()
    return df

### Create Table Summary

In [4]:
def generate_table_summary(df):
    """Menghasilkan ringkasan data bentuk tabel untuk Slide 1"""
    num_cols = df.select_dtypes(include=['number']).columns.tolist()
    text_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

    data_summary = []

    # Deteksi kolom revenue/price
    rev_col = next((col for col in df.columns if any(k in col.lower() for k in ['price', 'total', 'sales', 'amount', 'revenue'])), num_cols[0] if num_cols else None)
    total_rev = df[rev_col].sum() if rev_col else len(df)

    data_summary.append(["Total Volume Penjualan (Units)", "-", f"{len(df):,} Units", "100%", "EXCELLENT"])

    if rev_col:
        avg_rev = df[rev_col].mean()
        data_summary.append(["Total Value Penjualan (Revenue)", "Target Baseline", f"${total_rev:,.2f}", "105.4%", "EXCELLENT"])
        data_summary.append(["Rata-rata Harga Transaksi", "$25,000.00", f"${avg_rev:,.2f}", f"{(avg_rev/25000)*100:.1f}%", "ON TRACK" if avg_rev>=25000 else "NEEDS IMPR."])

    cat_col = next((col for col in text_cols if any(k in col.lower() for k in ['model', 'brand', 'make', 'category', 'region'])), text_cols[0] if text_cols else None)
    if cat_col:
        top_cat = df[cat_col].mode()[0] if not df[cat_col].empty else "N/A"
        top_cat_count = (df[cat_col] == top_cat).sum()
        pct_top = (top_cat_count / len(df)) * 100
        data_summary.append([f"Kontribusi Kategori Terlaris ({top_cat})", "20.00%", f"{pct_top:.2f}% ({top_cat_count:,} Units)", f"{(pct_top/20)*100:.1f}%", "EXCELLENT" if pct_top >= 20 else "ON TRACK"])

    return data_summary

In [5]:
def prepare_table_data(df):
    """
    Menghitung metrics dan periode sesuai dengan ketentuan poin 2 & 4.
    """
    # Pastikan Sale_Date berbentuk datetime
    df['Sale_Date'] = pd.to_datetime(df['Sale_Date'])

    # 4. Periode data dihitung berdasarkan min & max dari "Sale_Date"
    min_date = df['Sale_Date'].min()
    max_date = df['Sale_Date'].max()

    # Format Nama Bulan & Tahun (Contoh: "Januari 2023 - Juni 2024")
    periode_str = f"{min_date.strftime('%B %Y')} - {max_date.strftime('%B %Y')}"

    # 2. Perhitungan Metrics
    total_sales_unit = df['Sale_ID'].count()
    total_sales_price = df['Final_Sale_Price_LKR'].sum()
    total_car_brand = df['Car_Brand'].nunique()
    total_car_model = df['Car_Model'].nunique()
    avg_final_price = df['Final_Sale_Price_LKR'].mean()
    avg_discount_rate = df['Discount_Rate'].mean()

    # Penyesuaian format desimal / persentase pada Discount Rate
    formatted_discount = f"{avg_discount_rate * 100:.2f}%" if avg_discount_rate <= 1 else f"{avg_discount_rate:.2f}%"

    table_data = {
        "periode": periode_str,
        "metrics": [
            (1, "Total Sales Unit", f"{total_sales_unit:,}"),
            (2, "Total Sales Price", f"${total_sales_price:,.2f}"),
            (3, "Total Car Brand Sold", f"{total_car_brand:,}"),
            (4, "Total Car Model Sold", f"{total_car_model:,}"),
            (5, "Average Final Price per Car", f"${avg_final_price:,.2f}"),
            (6, "Average Discount Rate per Car", formatted_discount)
        ]
    }

    return table_data

### Create PPTX

#### Option 1 : Automatic Insight

In [6]:
def generate_gemini_dynamic_json(df_raw, api_key):
    """
    Mengirimkan metadata dataset baru ke Gemini API untuk mendapatkan
    Executive Summary & Rekomendasi Visualisasi Tren dalam bentuk JSON.
    """
    client = genai.Client(api_key=api_key)

    # 1. Ambil ringkasan struktur dataset secara otomatis
    num_cols = df_raw.select_dtypes(include=['number']).columns.tolist()
    date_cols = df_raw.select_dtypes(include=['datetime', 'datetimetz']).columns.tolist()
    if not date_cols:
        # Coba deteksi kolom tanggal dari string
        for col in df_raw.select_dtypes(include=['object']).columns:
            if any(k in col.lower() for k in ['date', 'tgl', 'time', 'bulan', 'month', 'tahun', 'year']):
                date_cols.append(col)
                break

    # Ringkasan statistik sederhana
    total_rows = len(df_raw)
    summary_dict = {"total_records": total_rows, "numeric_columns": {}}
    for col in num_cols[:5]:  # Ambil max 5 kolom angka utama
        summary_dict["numeric_columns"][col] = {
            "sum": float(df_raw[col].sum()),
            "mean": float(df_raw[col].mean())
        }

    date_col_name = date_cols[0] if date_cols else "N/A"

    prompt = f"""
    Anda adalah seorang Business Analyst Senior. Analisis metadata dataset berikut dan hasilkan respon HANYA dalam format JSON valid (tanpa teks penjelasan lain).

    Data Metadata:
    - Total Baris/Records: {total_rows}
    - Kolom Tanggal/Waktu Terdeteksi: {date_col_name}
    - Ringkasan Kolom Angka: {json.dumps(summary_dict)}

    Format JSON Output yang WAJIB dipatuhi:
    {{
      "title": "Judul Slide Analisis Performa",
      "executive_summary": [
        "- Poin analisis 1 mengenai total performa.",
        "- Poin analisis 2 mengenai tren atau pola utama.",
        "- Poin analisis 3 mengenai saran/rekomendasi strategis."
      ],
      "chart_config": {{
        "chart_title": "Nama Judul Tren Grafik",
        "date_column": "{date_col_name}",
        "value_column": "{num_cols[0] if num_cols else 'N/A'}"
      }}
    }}
    """

    candidate_models = ['gemini-2.0-flash', 'gemini-3.6-flash']
    max_retries = 3

    for model_name in candidate_models:
        for attempt in range(1, max_retries + 1):
            try:
                print(f"[INFO] Menghubungi Gemini API untuk Dataset Baru (Model: {model_name}, Attempt {attempt})...")
                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt
                )
                if response and response.text:
                    # Clean markdown code block if present
                    text = response.text.strip()
                    if "```json" in text:
                        text = re.search(r"```json(.*?)```", text, re.DOTALL).group(1).strip()
                    elif "```" in text:
                        text = re.search(r"```(.*?)```", text, re.DOTALL).group(1).strip()

                    return json.loads(text)
            except Exception as e:
                safe_err = str(e).encode('ascii', 'ignore').decode('ascii')
                print(f"[WARNING] Kendala pada {model_name}: {safe_err}")
                if "503" in str(e) or "UNAVAILABLE" in str(e):
                    time.sleep(2 ** attempt)
                else:
                    break

    # Fallback JSON jika API bermasalah/offline
    return {
        "title": "AI Executive Summary & Performance Trend",
        "executive_summary": [
            "- Laporan Performa: Penjualan dan transaksi secara umum menunjukkan kinerja stabil.",
            "- Catatan Tren: Terjadi pertumbuhan berkesinambungan pada beberapa periode utama.",
            "- Rekomendasi: Pertahankan efisiensi operasional pada cabang dengan performa tertinggi."
        ],
        "chart_config": {
            "chart_title": "Key Metrics Monthly Trend",
            "date_column": date_col_name,
            "value_column": num_cols[0] if num_cols else "N/A"
        }
    }


def create_slide_gemini_summary(prs, df_raw, api_key):
    """
    Membuat 1 slide terpadu secara OTOMATIS dan DINAMIS untuk data baru apapun:
    - Bagian Atas: Executive Summary (dari Gemini JSON)
    - Bagian Bawah: Chart Tren Otomatis berdasarkan kolom data baru
    """
    df_calc = df_raw.copy()

    # 1. Ambil Konfigurasi Dinamis dari Gemini API
    ai_data = generate_gemini_dynamic_json(df_calc, api_key)

    # 2. Inisialisasi Slide Presentation
    blank_layout = prs.slide_layouts[6]
    slide = prs.slides.add_slide(blank_layout)

    HEADER_BLUE = RGBColor(106, 127, 193)
    TEXT_DARK = RGBColor(30, 41, 59)
    CARD_BG = RGBColor(248, 250, 252)
    CARD_BORDER = RGBColor(226, 232, 240)
    BAR_BLUE = RGBColor(24, 144, 255)

    def add_card_header(left, top, width, height, title):
        card = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, left, top, width, height)
        card.fill.solid()
        card.fill.fore_color.rgb = CARD_BG
        card.line.color.rgb = CARD_BORDER

        header = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, left, top, width, Inches(0.35))
        header.fill.solid()
        header.fill.fore_color.rgb = HEADER_BLUE
        header.line.fill.background()

        tf = header.text_frame
        tf.word_wrap = True
        p = tf.paragraphs[0]
        p.text = title
        p.font.name = "Segoe UI"
        p.font.size = Pt(11)
        p.font.bold = True
        p.font.color.rgb = RGBColor(255, 255, 255)
        p.alignment = PP_ALIGN.CENTER
        return card

    # =========================================================
    # A. Judul Slide Dinamis
    # =========================================================
    title_box = slide.shapes.add_textbox(Inches(0.4), Inches(0.15), Inches(12.5), Inches(0.5))
    tf = title_box.text_frame
    p = tf.paragraphs[0]
    p.text = ai_data.get("title", "AI Executive Summary & Performance Trend")
    p.font.name = "Segoe UI"
    p.font.size = Pt(22)
    p.font.bold = True
    p.font.color.rgb = TEXT_DARK

    # =========================================================
    # B. BAGIAN ATAS: Executive Summary Box (Gemini AI)
    # =========================================================
    top_pos_y = Inches(0.75)
    box_width = Inches(12.533)
    box_height = Inches(2.4)

    add_card_header(Inches(0.4), top_pos_y, box_width, box_height, "Executive Summary & Strategic Insights (Powered by Gemini AI)")

    text_box = slide.shapes.add_textbox(Inches(0.5), top_pos_y + Inches(0.4), box_width - Inches(0.2), box_height - Inches(0.45))
    tf_text = text_box.text_frame
    tf_text.word_wrap = True
    tf_text.margin_left = Inches(0.2)
    tf_text.margin_right = Inches(0.2)
    tf_text.margin_top = Inches(0.1)

    exec_summaries = ai_data.get("executive_summary", [])
    for idx, line in enumerate(exec_summaries):
        p_line = tf_text.paragraphs[0] if idx == 0 else tf_text.add_paragraph()
        p_line.text = str(line)
        p_line.font.name = "Segoe UI"
        p_line.font.size = Pt(11)
        p_line.font.color.rgb = TEXT_DARK
        p_line.space_after = Pt(6)

    # =========================================================
    # C. BAGIAN BAWAH: Dynamic Sales/Metrics Trend Chart
    # =========================================================
    chart_pos_y = Inches(3.30)
    chart_height = Inches(3.80)

    chart_cfg = ai_data.get("chart_config", {})
    chart_title = chart_cfg.get("chart_title", "Key Metrics Performance Trend")
    date_col = chart_cfg.get("date_column")
    val_col = chart_cfg.get("value_column")

    add_card_header(Inches(0.4), chart_pos_y, box_width, chart_height, chart_title)

    # Pemrosesan Data Otomatis untuk Chart
    cdata = CategoryChartData()

    # Deteksi dan Agregasi Tanggal Dinamis
    if date_col and date_col in df_calc.columns and date_col != "N/A":
        try:
            df_calc['__Parsed_Date'] = pd.to_datetime(df_calc[date_col], errors='coerce')
            df_calc['__YearMonth'] = df_calc['__Parsed_Date'].dt.to_period('M')

            if val_col and val_col in df_calc.columns and val_col != "N/A":
                monthly_data = df_calc.groupby('__YearMonth')[val_col].sum().reset_index()
                series_vals = monthly_data[val_col].tolist()
            else:
                monthly_data = df_calc.groupby('__YearMonth').size().reset_index(name='Count')
                series_vals = monthly_data['Count'].tolist()

            monthly_data['MonthStr'] = monthly_data['__YearMonth'].dt.strftime('%b %Y')
            cdata.categories = monthly_data['MonthStr'].tolist()
            cdata.add_series('Metric Value', series_vals)
        except Exception:
            # Fallback jika parsing tanggal gagal
            counts = df_calc.iloc[:, 0].value_counts().head(10)
            cdata.categories = counts.index.astype(str).tolist()
            cdata.add_series('Volume', counts.values.tolist())
    else:
        # Fallback jika tidak ada kolom tanggal
        num_cols = df_calc.select_dtypes(include=['number']).columns
        if len(num_cols) > 0:
            sums = df_calc[num_cols[:8]].sum()
            cdata.categories = sums.index.astype(str).tolist()
            cdata.add_series('Total', sums.values.tolist())
        else:
            cdata.categories = ['Data']
            cdata.add_series('Total', [len(df_calc)])

    # Add Chart to Slide
    chart_shape = slide.shapes.add_chart(
        XL_CHART_TYPE.LINE_MARKERS,
        Inches(0.5), chart_pos_y + Inches(0.45),
        box_width - Inches(0.2), chart_height - Inches(0.55),
        cdata
    )
    chart = chart_shape.chart
    chart.has_title = False
    chart.has_legend = False

    series = chart.series[0]
    series.format.line.color.rgb = BAR_BLUE
    series.format.line.width = Pt(2.5)

    category_axis = chart.category_axis
    category_axis.tick_labels.font.size = Pt(8.5)
    category_axis.tick_labels.font.name = "Segoe UI"

    value_axis = chart.value_axis
    value_axis.tick_labels.font.size = Pt(8.5)
    value_axis.tick_labels.font.name = "Segoe UI"

    print("[SUCCESS] Dynamic Slide AI Executive Summary + Trend Chart berhasil dibuat!")

#### Option 2 : Table Summary



*   Total Sales Unit : count "Sale_ID"
*   Total Sales Price : sum "Final_Sale_Price_LKR"
*   Total Car Brand Sold : count distinct "Car_Brand"
*   Total Car Model Sold : count distinct "Car_Model"
*   Average Final Price per Car : average "Final_Sale_Price_LKR"
*   Average Discount Rate per Car : average "Discount_Rate"

In [7]:
# ---------------------------------------------------------
# 1. Fungsi Utama Sesuai Skema Permintaan
# ---------------------------------------------------------
def add_table_slide(prs, table_data):
    """
    Menambahkan slide baru berisikan tabel Key Performance Metrics
    dengan styling sesuai mockup terlampir.
    """
    # Menggunakan layout blank slide (Index 6)
    blank_slide_layout = prs.slide_layouts[6]
    slide = prs.slides.add_slide(blank_slide_layout)

    # --- A. Judul & Subtitle Slide ---
    title_box = slide.shapes.add_textbox(Inches(0.8), Inches(0.5), Inches(11.0), Inches(1.2))
    tf = title_box.text_frame
    tf.word_wrap = True
    tf.margin_left = tf.margin_top = tf.margin_right = tf.margin_bottom = 0

    # Title: Car Sales Performance
    p1 = tf.paragraphs[0]
    p1.text = "Car Sales Performance"
    p1.font.name = "Arial"
    p1.font.size = Pt(24)
    p1.font.bold = True
    p1.font.color.rgb = RGBColor(0, 0, 0)
    p1.space_after = Pt(4)

    # Subtitle: Performa selama periode bulan XX - XX
    p2 = tf.add_paragraph()
    p2.text = f"Performa selama periode bulan {table_data['periode']}"
    p2.font.name = "Arial"
    p2.font.size = Pt(16)
    p2.font.color.rgb = RGBColor(50, 50, 50)

    # --- B. Pembuatan Tabel ---
    rows = len(table_data['metrics']) + 1  # Header (1) + Data (6)
    cols = 3

    left = Inches(0.8)
    top = Inches(2.0)
    width = Inches(11.7)
    height = Inches(4.5)

    table_shape = slide.shapes.add_table(rows, cols, left, top, width, height)
    table = table_shape.table

    # Penyesuaian Lebar Kolom
    table.columns[0].width = Inches(0.8)   # Kolom '#'
    table.columns[1].width = Inches(6.4)   # Kolom 'Key Performance Metrics'
    table.columns[2].width = Inches(4.5)   # Kolom 'Achievement'

    # --- C. Header Styling (Warna Cerah Sesuai Mockup) ---
    headers = ["#", "Key Performance Metrics", "Achievement"]
    header_bg_color = RGBColor(14, 154, 206) # Cyan / Light Blue accent

    for col_idx, text in enumerate(headers):
        cell = table.cell(0, col_idx)
        cell.text = text
        cell.fill.solid()
        cell.fill.fore_color.rgb = header_bg_color

        # Formatting Teks Header
        p = cell.text_frame.paragraphs[0]
        p.font.name = "Arial"
        p.font.size = Pt(13)
        p.font.bold = True
        p.font.color.rgb = RGBColor(255, 255, 255) # Teks putih

        if col_idx == 0:
            p.alignment = PP_ALIGN.CENTER
        elif col_idx == 1:
            p.alignment = PP_ALIGN.CENTER
        else:
            p.alignment = PP_ALIGN.CENTER

    # --- D. Data Rows Styling (Alternating Zebra Striping) ---
    row_bg_1 = RGBColor(218, 233, 245) # Soft Ice Blue (Baris ganjil)
    row_bg_2 = RGBColor(235, 243, 250) # Light Ice Blue (Baris genap)

    for row_idx, data_tuple in enumerate(table_data['metrics'], start=1):
        bg_color = row_bg_1 if row_idx % 2 != 0 else row_bg_2

        for col_idx, val in enumerate(data_tuple):
            cell = table.cell(row_idx, col_idx)
            cell.text = str(val)
            cell.fill.solid()
            cell.fill.fore_color.rgb = bg_color

            p = cell.text_frame.paragraphs[0]
            p.font.name = "Arial"
            p.font.size = Pt(12)
            p.font.color.rgb = RGBColor(30, 30, 30)

            if col_idx == 0:
                p.alignment = PP_ALIGN.CENTER
            else:
                p.alignment = PP_ALIGN.LEFT

    return slide

#### Option 3 : Dashboard Summary

*   Total Sales Unit : count "Sale_ID"
*   Total Sales Price : sum "Final_Sale_Price_LKR"
*   Total Car Brand Sold : count distinct "Car_Brand"
*   Total Car Model Sold : count distinct "Car_Model"
*   Total Car Vehicle Year : count distinct "Vehicle_Year"
*   Total Sales Person : count distinct "Salesperson_ID"
*   Monthly Sales Trend : count "Sale_ID" and group by "Sale_Date" (monthly)
*   Top 3 Sales by Car Brand : top 3 count "Sale_ID" and group by "Car_Brand"
*   Top 3 Sales by Car Model : top 3 count "Sale_ID" and group by "Car_Model"
*   Transmission Type : count "Sale_ID" and group by "Transmission"
*   Payment Type : count "Sale_ID" and group by "Payment_Method"
*  Discount and Final Price : average "Discount_Rate" and average "Final_Sale_Price_LKR" and group by "Car_Model"

In [8]:
def prepare_table_data_option2(df):
    """
    Menghitung dan memformat seluruh metrics sesuai ketentuan.
    """
    df['Sale_Date'] = pd.to_datetime(df['Sale_Date'])

    total_sales_unit = df['Sale_ID'].count()
    total_sales_price = df['Final_Sale_Price_LKR'].sum()
    total_car_brand = df['Car_Brand'].nunique()
    total_car_model = df['Car_Model'].nunique()
    total_vehicle_year = df['Vehicle_Year'].nunique()
    total_sales_person = df['Salesperson_ID'].nunique()

    def fmt_num(val):
        if val >= 1e12:
            return f"{val/1e12:.1f}tn"
        elif val >= 1e9:
            v = val / 1e9
            return f"{v:.0f}bn" if abs(v - round(v)) < 0.05 else f"{v:.1f}bn"
        elif val >= 1e3:
            v = val / 1e3
            return f"{v:.0f}K" if abs(v - round(v)) < 0.05 else f"{v:.1f}K"
        return str(val)

    df['YearMonth'] = df['Sale_Date'].dt.to_period('M')
    monthly_trend = df.groupby('YearMonth')['Sale_ID'].count().reset_index()
    monthly_trend['MonthStr'] = monthly_trend['YearMonth'].dt.strftime('%b %Y')

    top_brands = df.groupby('Car_Brand')['Sale_ID'].count().nlargest(3).reset_index()
    top_models = df.groupby('Car_Model')['Sale_ID'].count().nlargest(3).reset_index()

    transmission = df.groupby('Transmission')['Sale_ID'].count().reset_index()
    payment = df.groupby('Payment_Method')['Sale_ID'].count().reset_index()

    disc_price = df.groupby('Car_Model').agg(
        Avg_Discount=('Discount_Rate', 'mean'),
        Avg_Final_Price=('Final_Sale_Price_LKR', 'mean')
    ).reset_index().sort_values(by='Car_Model')

    return {
        'kpis': [
            ("Total Sales Unit", fmt_num(total_sales_unit)),
            ("Total Sales Price", fmt_num(total_sales_price)),
            ("Total Car Brand", str(total_car_brand)),
            ("Total Car Model", str(total_car_model)),
            ("Total Car Vehicle Year", str(total_vehicle_year)),
            ("Total Sales Person", str(total_sales_person))
        ],
        'monthly_trend': monthly_trend,
        'top_brands': top_brands,
        'top_models': top_models,
        'transmission': transmission,
        'payment': payment,
        'disc_price': disc_price
    }


def create_slide_option2(prs, table_data):
    """
    Menambahkan slide Dashboard "Car Sales Performance" presisi sesuai mockup.
    """
    blank_layout = prs.slide_layouts[6]
    slide = prs.slides.add_slide(blank_layout)

    # Palet Warna
    HEADER_BLUE = RGBColor(106, 127, 193)  # #6A7FC1
    TEXT_DARK = RGBColor(35, 35, 35)
    CARD_BG = RGBColor(250, 250, 250)
    CARD_BORDER = RGBColor(230, 230, 230)
    BAR_BLUE = RGBColor(24, 144, 255)       # Bright Blue Accent (#1890FF)

    # Helper membuat Header Bar Card
    def add_card_header(left, top, width, height, title):
        card = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, left, top, width, height)
        card.fill.solid()
        card.fill.fore_color.rgb = CARD_BG
        card.line.color.rgb = CARD_BORDER

        header = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, left, top, width, Inches(0.35))
        header.fill.solid()
        header.fill.fore_color.rgb = HEADER_BLUE
        header.line.fill.background()

        tf = header.text_frame
        tf.word_wrap = True
        p = tf.paragraphs[0]
        p.text = title
        p.font.name = "Segoe UI"
        p.font.size = Pt(11)
        p.font.bold = True
        p.font.color.rgb = RGBColor(255, 255, 255)
        p.alignment = PP_ALIGN.CENTER
        return card

    # ==========================================
    # 1. Slide Title
    # ==========================================
    title_box = slide.shapes.add_textbox(Inches(0.4), Inches(0.15), Inches(10), Inches(0.5))
    tf = title_box.text_frame
    p = tf.paragraphs[0]
    p.text = "Car Sales Performance"
    p.font.name = "Segoe UI"
    p.font.size = Pt(22)
    p.font.bold = True
    p.font.color.rgb = TEXT_DARK

    # ==========================================
    # 2. Top 6 KPI Cards
    # ==========================================
    kpi_width = Inches(1.95)
    kpi_height = Inches(1.25)
    kpi_top = Inches(0.75)
    kpi_gap = Inches(0.10)
    start_left = Inches(0.4)

    for i, (title, val) in enumerate(table_data['kpis']):
        left = start_left + i * (kpi_width + kpi_gap)
        add_card_header(left, kpi_top, kpi_width, kpi_height, title)

        val_box = slide.shapes.add_textbox(left, kpi_top + Inches(0.4), kpi_width, Inches(0.8))
        tf_v = val_box.text_frame
        p_v = tf_v.paragraphs[0]
        p_v.text = str(val)
        p_v.font.name = "Segoe UI"
        p_v.font.size = Pt(32)
        p_v.font.bold = True
        p_v.font.color.rgb = TEXT_DARK
        p_v.alignment = PP_ALIGN.CENTER

    # ==========================================
    # 3. Monthly Sales Trend (Line Chart)
    # ==========================================
    trend_top = Inches(2.15)
    trend_width = Inches(12.2)
    trend_height = Inches(2.35)
    add_card_header(start_left, trend_top, trend_width, trend_height, "Monthly Sales Trend")

    cdata = CategoryChartData()
    df_trend = table_data['monthly_trend']
    cdata.categories = df_trend['MonthStr'].tolist()
    cdata.add_series('', df_trend['Sale_ID'].tolist())

    chart_shape = slide.shapes.add_chart(
        XL_CHART_TYPE.LINE_MARKERS,
        start_left + Inches(0.1), trend_top + Inches(0.4),
        trend_width - Inches(0.2), trend_height - Inches(0.45),
        cdata
    )
    chart = chart_shape.chart
    chart.has_title = False
    chart.has_legend = False

    series = chart.series[0]
    series.format.line.color.rgb = BAR_BLUE
    series.format.line.width = Pt(2.5)

    # Format ukuran font X-Axis dan Y-Axis
    category_axis = chart.category_axis
    category_axis.tick_labels.font.size = Pt(7.5)
    category_axis.tick_labels.font.name = "Segoe UI"

    value_axis = chart.value_axis
    value_axis.tick_labels.font.size = Pt(7.5)
    value_axis.tick_labels.font.name = "Segoe UI"

    # ==========================================
    # 4. Bottom Row 1: Top 3 Sales by Car Brand
    # ==========================================
    bot_top = Inches(4.60)
    bot_height = Inches(2.65)

    brand_width = Inches(1.85)
    add_card_header(start_left, bot_top, brand_width, bot_height, "Top 3 Sales by Car Brand")

    cdata_b = CategoryChartData()
    df_b = table_data['top_brands']
    cdata_b.categories = df_b['Car_Brand'].tolist()
    cdata_b.add_series('', df_b['Sale_ID'].tolist())

    chart_b = slide.shapes.add_chart(
        XL_CHART_TYPE.COLUMN_CLUSTERED,
        start_left + Inches(0.05), bot_top + Inches(0.4),
        brand_width - Inches(0.1), bot_height - Inches(0.45),
        cdata_b
    ).chart

    chart_b.has_title = False
    chart_b.has_legend = False
    chart_b.value_axis.visible = False
    chart_b.value_axis.has_major_gridlines = False

    series_b = chart_b.series[0]
    series_b.format.fill.solid()
    series_b.format.fill.fore_color.rgb = BAR_BLUE

    plots_b = chart_b.plots[0]
    plots_b.has_data_labels = True
    data_labels_b = plots_b.data_labels
    data_labels_b.position = XL_LABEL_POSITION.OUTSIDE_END
    data_labels_b.font.size = Pt(8.5)
    data_labels_b.font.name = "Segoe UI"
    data_labels_b.font.bold = True

    # ==========================================
    # 5. Bottom Row 2: Top 3 Sales by Car Model
    # ==========================================
    model_left = start_left + brand_width + Inches(0.12)
    model_width = Inches(1.85)
    add_card_header(model_left, bot_top, model_width, bot_height, "Top 3 Sales by Car Model")

    cdata_m = CategoryChartData()
    df_m = table_data['top_models']
    cdata_m.categories = df_m['Car_Model'].tolist()
    cdata_m.add_series('', df_m['Sale_ID'].tolist())

    chart_m = slide.shapes.add_chart(
        XL_CHART_TYPE.COLUMN_CLUSTERED,
        model_left + Inches(0.05), bot_top + Inches(0.4),
        model_width - Inches(0.1), bot_height - Inches(0.45),
        cdata_m
    ).chart

    chart_m.has_title = False
    chart_m.has_legend = False
    chart_m.value_axis.visible = False
    chart_m.value_axis.has_major_gridlines = False

    series_m = chart_m.series[0]
    series_m.format.fill.solid()
    series_m.format.fill.fore_color.rgb = BAR_BLUE

    plots_m = chart_m.plots[0]
    plots_m.has_data_labels = True
    data_labels_m = plots_m.data_labels
    data_labels_m.position = XL_LABEL_POSITION.OUTSIDE_END
    data_labels_m.font.size = Pt(8.5)
    data_labels_m.font.name = "Segoe UI"
    data_labels_m.font.bold = True

    # ==========================================
    # 6. Bottom Row 3: Transmission Type (Doughnut Chart)
    # ==========================================
    trans_left = model_left + model_width + Inches(0.12)
    trans_width = Inches(2.05)
    add_card_header(trans_left, bot_top, trans_width, bot_height, "Transmission Type")

    cdata_t = CategoryChartData()
    df_t = table_data['transmission']
    cdata_t.categories = df_t['Transmission'].tolist()
    cdata_t.add_series('', df_t['Sale_ID'].tolist())

    chart_t = slide.shapes.add_chart(
        XL_CHART_TYPE.DOUGHNUT,
        trans_left + Inches(0.05), bot_top + Inches(0.4),
        trans_width - Inches(0.1), bot_height - Inches(0.45),
        cdata_t
    ).chart

    chart_t.has_title = False
    chart_t.has_legend = True
    chart_t.legend.position = XL_LEGEND_POSITION.BOTTOM
    chart_t.legend.include_in_layout = False

    plots_t = chart_t.plots[0]
    plots_t.has_data_labels = True
    data_labels_t = plots_t.data_labels
    data_labels_t.number_format = '0.0%'
    data_labels_t.show_percentage = True
    data_labels_t.show_value = False
    data_labels_t.font.size = Pt(8.5)
    data_labels_t.font.name = "Segoe UI"
    data_labels_t.font.bold = True

    # ==========================================
    # 7. Bottom Row 4: Payment Type (Pie Chart)
    # ==========================================
    pay_left = trans_left + trans_width + Inches(0.12)
    pay_width = Inches(2.20)
    add_card_header(pay_left, bot_top, pay_width, bot_height, "Payment Type")

    cdata_p = CategoryChartData()
    df_p = table_data['payment']
    cdata_p.categories = df_p['Payment_Method'].tolist()
    cdata_p.add_series('', df_p['Sale_ID'].tolist())

    chart_p = slide.shapes.add_chart(
        XL_CHART_TYPE.PIE,
        pay_left + Inches(0.05), bot_top + Inches(0.4),
        pay_width - Inches(0.1), bot_height - Inches(0.45),
        cdata_p
    ).chart

    chart_p.has_title = False
    chart_p.has_legend = True
    chart_p.legend.position = XL_LEGEND_POSITION.BOTTOM
    chart_p.legend.include_in_layout = False

    plots_p = chart_p.plots[0]
    plots_p.has_data_labels = True
    data_labels_p = plots_p.data_labels
    data_labels_p.number_format = '0.0%'
    data_labels_p.show_percentage = True
    data_labels_p.show_value = False
    data_labels_p.font.size = Pt(8.5)
    data_labels_p.font.name = "Segoe UI"
    data_labels_p.font.bold = True

    # ==========================================
    # 8. Bottom Row 5: Discount and Final Price (Table)
    # ==========================================
    table_left = pay_left + pay_width + Inches(0.12)
    table_width = Inches(3.85)
    add_card_header(table_left, bot_top, table_width, bot_height, "Discount and Final Price")

    df_dp = table_data['disc_price']
    rows = min(len(df_dp) + 1, 7)
    cols = 3

    t_shape = slide.shapes.add_table(
        rows, cols,
        table_left + Inches(0.1), bot_top + Inches(0.45),
        table_width - Inches(0.2), Inches(2.0)
    )
    tbl = t_shape.table
    tbl.columns[0].width = Inches(1.25)
    tbl.columns[1].width = Inches(1.10)
    tbl.columns[2].width = Inches(1.30)

    headers = ["Car_Model", "Avg Discount", "Avg Final Price"]
    for j, h in enumerate(headers):
        cell = tbl.cell(0, j)
        cell.text = h
        p = cell.text_frame.paragraphs[0]
        p.font.name = "Segoe UI"
        p.font.size = Pt(9.5)
        p.font.bold = True
        p.font.color.rgb = TEXT_DARK
        p.alignment = PP_ALIGN.RIGHT if j > 0 else PP_ALIGN.LEFT

    for i in range(1, rows):
        row_data = df_dp.iloc[i - 1]
        vals = [
            f"⊞ {row_data['Car_Model']}",
            f"{row_data['Avg_Discount']:.2f}",
            f"{row_data['Avg_Final_Price']:,.2f}"
        ]

        bg_col = RGBColor(250, 250, 250) if i % 2 != 0 else RGBColor(240, 243, 248)

        for j, v in enumerate(vals):
            cell = tbl.cell(i, j)
            cell.text = v
            cell.fill.solid()
            cell.fill.fore_color.rgb = bg_col

            p = cell.text_frame.paragraphs[0]
            p.font.name = "Segoe UI"
            p.font.size = Pt(9)
            p.font.color.rgb = TEXT_DARK
            p.alignment = PP_ALIGN.RIGHT if j > 0 else PP_ALIGN.LEFT

    return slide

### Convert PPTX to PDF

In [9]:
def convert_pptx_to_pdf(input_pptx, output_pdf):
    """Konversi PPTX ke PDF lintas platform (Windows / Linux / LibreOffice / Aspose)"""
    print("Mulai konversi PPTX ke PDF...")
    current_os = platform.system()
    input_abs = os.path.abspath(input_pptx)
    output_abs = os.path.abspath(output_pdf)

    # Windows COM Method
    if current_os == "Windows":
        try:
            import comtypes.client
            powerpoint = comtypes.client.CreateObject("PowerPoint.Application")
            powerpoint.Visible = 1
            presentation = powerpoint.Presentations.Open(input_abs)
            presentation.SaveAs(output_abs, 32)
            presentation.Close()
            powerpoint.Quit()
            print(f"✅ Konversi Windows PowerPoint berhasil! PDF: {output_abs}")
            return True
        except Exception as e:
            print(f"Windows COM error: {e}")

    # LibreOffice Method
    try:
        libreoffice_path = "soffice" if current_os == "Windows" else ("libreoffice" if current_os != "Darwin" else "/Applications/LibreOffice.app/Contents/MacOS/soffice")
        cmd = [libreoffice_path, "--headless", "--convert-to", "pdf", "--outdir", os.path.dirname(output_abs), input_abs]
        subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        print(f"✅ Konversi LibreOffice sukses! PDF: {output_abs}")
        return True
    except Exception:
        pass

    return False

### Send Email

In [10]:
def send_report_email(sender_email, sender_password, receiver_email, file_paths):
    print("\nMenyiapkan pengiriman email...")
    smtp_server = "smtp.gmail.com"
    smtp_port = 587

    msg = MIMEMultipart()
    msg['From'] = sender_email
    msg['To'] = receiver_email
    msg['Subject'] = "📊 Laporan Performa Car Sales Dashboard"

    body = "Halo Team,\n\nBerikut kami lampirkan Laporan Performa Penjualan Mobil (Car Sales OLTP Dataset) hasil otomasi Python.\n\nSalam,\nAutomation Bot"
    msg.attach(MIMEText(body, 'plain'))

    for filepath in file_paths:
        if os.path.exists(filepath):
            with open(filepath, "rb") as attachment:
                part = MIMEBase("application", "octet-stream")
                part.set_payload(attachment.read())
            encoders.encode_base64(part)
            part.add_header("Content-Disposition", f"attachment; filename= {os.path.basename(filepath)}")
            msg.attach(part)

    try:
        server = smtplib.SMTP(smtp_server, smtp_port)
        server.starttls()
        server.login(sender_email, sender_password)
        server.sendmail(sender_email, receiver_email, msg.as_string())
        print("✅ Email sukses terkirim ke:", receiver_email)
    except Exception as e:
        print("❌ Gagal mengirimkan email. Error:", e)
    finally:
        server.quit()

In [11]:
def get_email_mapping():
    """
    Mengambil mapping cabang dan daftar email berdasarkan Opsi 1 atau Opsi 2.
    Output: dict -> {"Branch A": ["email1@test.com", "email2@test.com"], ...}
    """
    print("\n" + "="*50)
    print("PILIH METODE PENENTUAN PENERIMA EMAIL:")
    print("1. Input Manual (Branch Name & Email)")
    print("2. Upload File Excel List Penerima")
    print("="*50)

    opsi = input("Masukkan opsi (1/2) [Default: 1]: ").strip() or "1"
    branch_email_map = {}

    if opsi == "1":
        print("\n--- OPSI 1: INPUT MANUAL ---")
        while True:
            branch = input("Masukkan Nama Cabang (atau tekan Enter untuk selesai): ").strip()
            if not branch:
                break
            emails_raw = input(f"Masukkan Email penerima untuk {branch} (pisahkan dengan ';'): ").strip()
            # Split berdasarkan ';' dan bersihkan whitespace
            emails = [e.strip() for e in emails_raw.split(";") if e.strip()]

            if branch in branch_email_map:
                branch_email_map[branch].extend(emails)
            else:
                branch_email_map[branch] = emails

    elif opsi == "2":
        print("\n--- OPSI 2: UPLOAD FILE EXCEL ---")
        excel_path = input("Masukkan path/nama file Excel (contoh: list_user_email.xlsx): ").strip()
        try:
            df_excel = pd.read_excel(excel_path)
            # Asumsi kolom 'Branch_Name' dan 'Email' (bisa disesuaikan dengan header file excel Anda)
            col_branch = next((col for col in df_excel.columns if 'branch' in col.lower()), df_excel.columns[0])
            col_email = next((col for col in df_excel.columns if 'email' in col.lower()), df_excel.columns[1])

            for _, row in df_excel.iterrows():
                branch = str(row[col_branch]).strip()
                emails_raw = str(row[col_email]).strip()
                emails = [e.strip() for e in emails_raw.split(";") if e.strip() and e.strip().lower() != 'nan']

                if branch in branch_email_map:
                    branch_email_map[branch].extend(emails)
                else:
                    branch_email_map[branch] = emails
            print(f"✅ Berhasil memuat daftar email dari Excel untuk {len(branch_email_map)} cabang.")
        except Exception as e:
            print(f"❌ Gagal membaca file Excel: {e}")

    return branch_email_map


def generate_report_for_branch(df_raw, branch_name, pilihan_slide):
    """
    Filter data berdasarkan branch_name dan generate file PPTX & PDF khusus cabang tersebut.
    """
    # Asumsi nama kolom cabang pada dataset (contoh: 'Branch_Name', 'Branch', atau 'Region')
    col_branch_ds = next((col for col in df_raw.columns if 'branch' in col.lower() or 'region' in col.lower()), None)

    if col_branch_ds and col_branch_ds in df_raw.columns:
        df_branch = df_raw[df_raw[col_branch_ds].astype(str).str.upper() == branch_name.upper()].copy()
        if df_branch.empty:
            print(f"⚠️ Warning: Data untuk cabang '{branch_name}' tidak ditemukan pada dataset! Menggunakan seluruh data sebagai fallback.")
            df_branch = df_raw.copy()
    else:
        print(f"⚠️ Kolom 'Branch' tidak ditemukan pada dataset, laporan dibuat menggunakan seluruh data.")
        df_branch = df_raw.copy()

    # Hitung ulang metrics data terfilter
    table_data_option1 = prepare_table_data(df_branch)
    table_data_option2 = prepare_table_data_option2(df_branch)

    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)

    if pilihan_slide == "1":
        add_table_slide(prs, table_data_option1)
    elif pilihan_slide == "2":
        create_slide_option2(prs, table_data_option2)
    else:
        add_table_slide(prs, table_data_option1)
        create_slide_option2(prs, table_data_option2)

    clean_branch_name = branch_name.replace(" ", "_")
    file_pptx = f"Laporan_Performa_{clean_branch_name}.pptx"
    file_pdf = f"Laporan_Performa_{clean_branch_name}.pdf"

    prs.save(file_pptx)
    konversi_sukses = convert_pptx_to_pdf(file_pptx, file_pdf)

    files_to_send = [file_pptx]
    if konversi_sukses and os.path.exists(file_pdf):
        files_to_send.append(file_pdf)

    return files_to_send


def send_report_email_multi(sender_email, sender_password, receiver_emails, file_paths, branch_name):
    """
    Mengirimkan email laporan ke multiple penerima untuk cabang tertentu.
    """
    if not receiver_emails:
        print(f"⚠️ Tidak ada email penerima untuk cabang {branch_name}.")
        return

    print(f"\nMenyiapkan pengiriman email cabang {branch_name} ke: {', '.join(receiver_emails)}...")
    smtp_server = "smtp.gmail.com"
    smtp_port = 587

    msg = MIMEMultipart()
    msg['From'] = sender_email
    msg['To'] = ", ".join(receiver_emails)
    msg['Subject'] = f"📊 Laporan Performa Car Sales Dashboard - Cabang {branch_name}"

    body = f"Halo Team Cabang {branch_name},\n\nBerikut kami lampirkan Laporan Performa Penjualan Mobil khusus untuk cabang {branch_name}.\n\nSalam,\nAutomation Bot"
    msg.attach(MIMEText(body, 'plain'))

    for filepath in file_paths:
        if os.path.exists(filepath):
            with open(filepath, "rb") as attachment:
                part = MIMEBase("application", "octet-stream")
                part.set_payload(attachment.read())
            encoders.encode_base64(part)
            part.add_header("Content-Disposition", f"attachment; filename= {os.path.basename(filepath)}")
            msg.attach(part)

    try:
        server = smtplib.SMTP(smtp_server, smtp_port)
        server.starttls()
        server.login(sender_email, sender_password)
        server.sendmail(sender_email, receiver_emails, msg.as_string())
        print(f"✅ Email cabang {branch_name} sukses terkirim!")
    except Exception as e:
        print(f"❌ Gagal mengirimkan email cabang {branch_name}. Error: {e}")
    finally:
        server.quit()

## Main Query

In [13]:
if __name__ == "__main__":
    csv_sample_url = "https://github.com/nurchamid/ReportAutomation/blob/main/Car_Sales_OLTP_SLStyle_18Months.csv"
    file_pptx = "Laporan_Performa_Car_Sales.pptx"
    file_pdf = "Laporan_Performa_Car_Sales.pdf"

    # 1. Olah data CSV
    df_raw = load_and_process_car_sales_data(csv_sample_url)
    # table_data = generate_table_summary(df_raw)
    table_data_option1 = prepare_table_data(df_raw)
    table_data_option2 = prepare_table_data_option2(df_raw)

    # 2. Pilihan Jenis Slide Laporan
    print("\n" + "="*50)
    print("PILIH LAYOUT SLIDE LAPORAN YANG INGIN DIBUAT:")
    print("1. AI Executive Summary (Generated by AI)")
    print("2. Tabel Performa Executive (Slide KPI Standar)")
    print("3. Visual Dashboard (Card & Grafik ala Power BI)")
    print("4. Lengkap (Tabel, Dashboard, & Gemini AI Summary)")
    print("="*50)

    pilihan_slide = input("Masukkan nomor pilihan (1-4) [Default: 2]: ").strip() or "2"

    gemini_key = None
    if pilihan_slide in ["1", "4"]:
        gemini_key = getpass("Masukkan Gemini API Key Anda: ").strip()

    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)

    if pilihan_slide == "1":
        create_slide_gemini_summary(prs, df_raw, gemini_key)
        print("Slide AI Executive Summary (Gemini API) berhasil ditambahkan.")
    elif pilihan_slide == "2":
        add_table_slide(prs, table_data_option1)
        print("Slide 1 (Tabel Performa) berhasil ditambahkan.")
    elif pilihan_slide == "3":
        create_slide_option2(prs, table_data_option2)
        print("Slide 2 (Visual Dashboard Power BI) berhasil ditambahkan.")
    elif pilihan_slide == "4":
        create_slide_gemini_summary(prs, df_raw, gemini_key)
        add_table_slide(prs, table_data_option1)
        create_slide_option2(prs, table_data_option2)
        print("Slide Lengkap (Tabel, Dashboard, & Gemini AI Summary) berhasil ditambahkan.")

    prs.save(file_pptx)
    print(f"✅ File PPTX disimpan di: {os.path.abspath(file_pptx)}")

    # 3. Konversi ke PDF
    konversi_sukses = convert_pptx_to_pdf(file_pptx, file_pdf)

    # 4. Pengiriman Email (Opsional)
    print("\n" + "="*40)
    pilihan_email = input("Apakah Anda ingin mengirimkan laporan ini ke email? (y/n): ").lower().strip()

    if pilihan_email == 'y':
        sender = input("Masukkan email pengirim (Gmail): ").strip()
        password = getpass("Masukkan App Password pengirim: ")
        receiver = input("Masukkan email penerima: ").strip()

        files_to_send = [file_pptx]
        if konversi_sukses and os.path.exists(file_pdf):
            files_to_send.append(file_pdf)

        send_report_email(sender, password, receiver, files_to_send)
    else:
        print("Proses selesai tanpa pengiriman email. File disimpan secara lokal.")

Membaca data dari: https://raw.githubusercontent.com/nurchamid/ReportAutomation/main/Car_Sales_OLTP_SLStyle_18Months.csv
✅ Data berhasil dimuat! Total baris: 50,000

PILIH LAYOUT SLIDE LAPORAN YANG INGIN DIBUAT:
1. AI Executive Summary (Generated by AI)
2. Tabel Performa Executive (Slide KPI Standar)
3. Visual Dashboard (Card & Grafik ala Power BI)
4. Lengkap (Tabel, Dashboard, & Gemini AI Summary)
Masukkan nomor pilihan (1-4) [Default: 2]: 4
Masukkan Gemini API Key Anda: ··········
[INFO] Menghubungi Gemini API untuk Dataset Baru (Model: gemini-2.0-flash, Attempt 1)...
[WARNING] Kendala pada gemini-2.0-flash: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'status': 'NOT_FOUND'}}
[INFO] Menghubungi Gemini API untuk Dataset Baru (Model: gemini-3.6-flash, Attempt 1)...
[SUCCESS] Dyna